# Explorar datos por deporte

Template papermill para exploración inicial de un deporte.

**Uso**: `make notebook NB=01_explore_sport -p sport nba -p seasons 2024,2025`

In [ ]:
sport = 'nba'
seasons = '2024,2025'
limit_rows = 10_000

In [ ]:
import asyncio
import polars as pl
from sqlalchemy import text
from apuestas.db import session_scope

async def load_matches(sport_code: str, seasons_csv: str, n: int) -> pl.DataFrame:
    seasons_list = [s.strip() for s in seasons_csv.split(',')]
    async with session_scope() as s:
        result = await s.execute(
            text('SELECT * FROM matches WHERE sport_code = :sp AND season = ANY(:se) LIMIT :n'),
            {'sp': sport_code, 'se': seasons_list, 'n': n},
        )
        return pl.DataFrame([dict(r._mapping) for r in result.all()])

df = asyncio.run(load_matches(sport, seasons, limit_rows))
print(f'Matches cargados: {df.height}')
df.head()

In [ ]:
# Distribución de goles / scores por partido
if not df.is_empty() and 'home_score' in df.columns:
    totals = (df['home_score'].fill_null(0) + df['away_score'].fill_null(0))
    print(f'Avg total: {totals.mean():.2f} | std: {totals.std():.2f}')
    print(f'Home wins: {(df["home_score"] > df["away_score"]).sum()} / {df.height}')

In [ ]:
# Home advantage empírico
if not df.is_empty() and 'home_score' in df.columns:
    hw_rate = (df['home_score'] > df['away_score']).mean()
    print(f'Home win rate: {hw_rate:.4f}  (expected ~0.55-0.60 según deporte)')